# 01 Macro-Bias Quantification

Run the six balanced models (3 CF + 3 CBF) on one dataset split and quantify popularity/exposure bias.

## 1) Setup and Imports

In [ ]:
import gc
import os
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline

from surprise import Dataset, Reader, KNNBasic, SVD

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_ROOT = ROOT / 'data'
FIG_DIR = ROOT / 'outputs' / 'figures'
RES_DIR = ROOT / 'outputs' / 'results'
FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

import sys
sys.path.append(str(ROOT / 'src'))
from utils import load_dataset, ensure_output_dirs
ensure_output_dirs(ROOT)

RANDOM_STATE = 42
TOP_K = 10


## 2) Config

In [ ]:
DATASET_NAME = 'ml-100k'  # options: ml-100k, ml-1m, lastfm-2k, book-crossing
TEST_SIZE = 0.2


## 3) Load Data and Standardize

In [ ]:
ratings, item_features = load_dataset(
    dataset_name=DATASET_NAME,
    data_root=DATA_ROOT,
    lastfm_mode='1-5',
    apply_k_core=True,
    min_user_interactions=10,
    min_item_interactions=10,
)

ratings = ratings.dropna().reset_index(drop=True)
print(ratings.head())
print(item_features.head())
print(f'Interactions: {len(ratings):,}, Users: {ratings.user_id.nunique():,}, Items: {ratings.item_id.nunique():,}')


## 4) Split Train/Test

In [ ]:
train_df, test_df = train_test_split(ratings, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

all_items = sorted(train_df['item_id'].unique().tolist())
users_test = sorted(test_df['user_id'].unique().tolist())
user_seen = build_user_seen(train_df)
train_pop = popularity_series(train_df)
pop_map = train_pop.to_dict()


## 5) Helper Functions

In [ ]:
def popularity_series(df):
    return df['item_id'].value_counts().astype(float)


def gini_coefficient(x):
    x = np.asarray(x, dtype=float)
    if x.size == 0:
        return np.nan
    if np.amin(x) < 0:
        x = x - np.amin(x)
    x = x + 1e-9
    x = np.sort(x)
    n = x.size
    index = np.arange(1, n + 1)
    return (np.sum((2 * index - n - 1) * x)) / (n * np.sum(x))


def ndcg_at_k(recommended_items, relevant_items, k=10):
    rel_set = set(relevant_items)
    dcg = 0.0
    for i, it in enumerate(recommended_items[:k], start=1):
        if it in rel_set:
            dcg += 1.0 / np.log2(i + 1)
    ideal_hits = min(len(rel_set), k)
    if ideal_hits == 0:
        return 0.0
    idcg = np.sum([1.0 / np.log2(i + 1) for i in range(1, ideal_hits + 1)])
    return float(dcg / idcg)


def compute_catalog_coverage(recs, all_items):
    if not recs:
        return 0.0
    rec_items = set([i for items in recs.values() for i in items])
    return len(rec_items) / max(1, len(all_items))


def compute_arp(recs, pop_map):
    vals = []
    for _, items in recs.items():
        vals.extend([pop_map.get(i, 0.0) for i in items])
    return float(np.mean(vals)) if vals else 0.0


def train_popularity_percentile_map(train_df):
    vc = train_df['item_id'].value_counts()
    return vc.rank(pct=True, ascending=False).to_dict()


def avg_popularity_percentile(recs, pct_map):
    vals = [pct_map.get(it, np.nan) for items in recs.values() for it in items]
    vals = [v for v in vals if np.isfinite(v)]
    return float(np.mean(vals)) if vals else np.nan


def long_tail_share(recs, pct_map, tail_cutoff=0.2):
    total, tail = 0, 0
    for items in recs.values():
        for it in items:
            total += 1
            p = pct_map.get(it, np.nan)
            if np.isfinite(p) and p <= tail_cutoff:
                tail += 1
    return float(tail / total) if total else 0.0


def aggregate_diversity_unique_items(recs):
    return len({it for items in recs.values() for it in items})


def metadata_token_entropy(recs, item_features):
    meta = dict(zip(item_features['item_id'].astype(str), item_features['metadata_text'].fillna('')))
    c = Counter()
    for items in recs.values():
        for it in items:
            for tok in str(meta.get(str(it), '')).lower().split():
                if tok:
                    c[tok] += 1
    total = sum(c.values())
    if total == 0:
        return np.nan
    return float(-sum((n / total) * np.log2(n / total) for n in c.values()))


def recommendation_frequency(recs):
    freq = {}
    for _, items in recs.items():
        for it in items:
            freq[it] = freq.get(it, 0) + 1
    return pd.Series(freq, dtype=float)

def build_user_seen(train_df):
    return train_df.groupby('user_id')['item_id'].apply(set).to_dict()


def surprise_prepare(train_df):
    reader = Reader(rating_scale=(float(train_df['rating'].min()), float(train_df['rating'].max())))
    data = Dataset.load_from_df(train_df[['user_id', 'item_id', 'rating']], reader)
    trainset = data.build_full_trainset()
    return trainset


def get_candidate_items(all_items, seen_set):
    if not seen_set:
        return all_items
    return [i for i in all_items if i not in seen_set]


def recommend_with_surprise(algo, users, user_seen, all_items, k=10):
    recs = {}
    for u in tqdm(users, desc='Surprise Top-K'):
        candidates = get_candidate_items(all_items, user_seen.get(u, set()))
        preds = [(it, algo.predict(u, it).est) for it in candidates]
        preds.sort(key=lambda x: x[1], reverse=True)
        recs[u] = [it for it, _ in preds[:k]]
    return recs


def rmse_surprise(algo, test_df):
    preds = [algo.predict(r.user_id, r.item_id).est for r in test_df.itertuples(index=False)]
    return float(np.sqrt(mean_squared_error(test_df['rating'].values, preds)))


def tfidf_item_similarity(item_features):
    tfidf = TfidfVectorizer(min_df=1, max_features=30000)
    mat = tfidf.fit_transform(item_features['metadata_text'])
    sim = cosine_similarity(mat, dense_output=False)
    return sim


def recommend_tfidf(train_df, users, all_items, item_index, sim_matrix, k=10):
    user_hist = train_df.groupby('user_id')['item_id'].apply(list).to_dict()
    recs = {}
    for u in tqdm(users, desc='TF-IDF Top-K'):
        seen = set(user_hist.get(u, []))
        if not seen:
            recs[u] = all_items[:k]
            continue
        seen_idx = [item_index[it] for it in seen if it in item_index]
        if not seen_idx:
            recs[u] = all_items[:k]
            continue
        profile_scores = sim_matrix[seen_idx].mean(axis=0).A1
        scored_items = []
        for it in all_items:
            if it in seen:
                continue
            idx = item_index.get(it)
            if idx is not None:
                scored_items.append((it, profile_scores[idx]))
        scored_items.sort(key=lambda x: x[1], reverse=True)
        recs[u] = [it for it, _ in scored_items[:k]]
    return recs


def train_user_logistic_models(train_df, item_features, user_min_pos=5):
    merged = train_df.merge(item_features, on='item_id', how='left')
    merged['label'] = (merged['rating'] >= merged['rating'].median()).astype(int)
    models = {}
    for user_id, grp in tqdm(merged.groupby('user_id'), desc='LogReg per-user'):
        if grp['label'].nunique() < 2 or len(grp) < user_min_pos:
            continue
        X = grp['metadata_text'].fillna('')
        y = grp['label']
        pipe = Pipeline([
            ('tfidf', TfidfVectorizer(min_df=1, max_features=5000)),
            ('clf', LogisticRegression(max_iter=300))
        ])
        pipe.fit(X, y)
        models[user_id] = pipe
    return models


def recommend_user_logreg(models, users, user_seen, item_features, all_items, k=10):
    item_text_map = dict(zip(item_features['item_id'], item_features['metadata_text']))
    recs = {}
    for u in tqdm(users, desc='LogReg Top-K'):
        seen = user_seen.get(u, set())
        candidates = [it for it in all_items if it not in seen]
        m = models.get(u)
        if m is None:
            recs[u] = candidates[:k]
            continue
        X = [item_text_map.get(it, '') for it in candidates]
        if len(X) == 0:
            recs[u] = []
            continue
        probs = m.predict_proba(X)[:, 1]
        order = np.argsort(-probs)[:k]
        recs[u] = [candidates[i] for i in order]
    return recs


def train_rf_regressor(train_df):
    df = train_df.copy()
    ue = LabelEncoder()
    ie = LabelEncoder()
    df['u'] = ue.fit_transform(df['user_id'])
    df['i'] = ie.fit_transform(df['item_id'])
    X = df[['u', 'i']]
    y = df['rating']
    rf = RandomForestRegressor(
        n_estimators=120,
        max_depth=18,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    )
    rf.fit(X, y)
    return rf, ue, ie


def recommend_rf(rf, ue, ie, users, user_seen, all_items, k=10):
    item_to_enc = {it: idx for idx, it in enumerate(ie.classes_)}
    recs = {}
    for u in tqdm(users, desc='RF Top-K'):
        seen = user_seen.get(u, set())
        candidates = [it for it in all_items if it not in seen and it in item_to_enc]
        if len(candidates) == 0 or u not in set(ue.classes_):
            recs[u] = all_items[:k]
            continue
        u_enc = ue.transform([u])[0]
        X = pd.DataFrame({'u': [u_enc] * len(candidates), 'i': [item_to_enc[it] for it in candidates]})
        preds = rf.predict(X)
        order = np.argsort(-preds)[:k]
        recs[u] = [candidates[i] for i in order]
    return recs


## 6) Train Models and Generate Top-10

In [ ]:
model_recs = {}
model_rmse = {}

# ----- CF: k-NN -----
trainset = surprise_prepare(train_df)
knn = KNNBasic(sim_options={'name': 'cosine', 'user_based': True}, verbose=False)
knn.fit(trainset)
model_recs['k-NN'] = recommend_with_surprise(knn, users_test, user_seen, all_items, k=TOP_K)
model_rmse['k-NN'] = rmse_surprise(knn, test_df)
del knn
gc.collect()

# ----- CF: SVD -----
svd = SVD(random_state=RANDOM_STATE)
svd.fit(trainset)
model_recs['SVD'] = recommend_with_surprise(svd, users_test, user_seen, all_items, k=TOP_K)
model_rmse['SVD'] = rmse_surprise(svd, test_df)
del svd
gc.collect()

# ----- CF: ALS (Implicit) -----
try:
    from scipy.sparse import coo_matrix
    from implicit.als import AlternatingLeastSquares

    train_users = sorted(train_df['user_id'].unique().tolist())
    item_to_idx = {it: j for j, it in enumerate(all_items)}

    rows = pd.Categorical(train_df['item_id'], categories=all_items).codes.astype(np.int64)
    cols = pd.Categorical(train_df['user_id'], categories=train_users).codes.astype(np.int64)
    if (rows < 0).any() or (cols < 0).any():
        raise ValueError('ALS indexing failed: found unknown user_id/item_id codes during matrix build.')

    vals = train_df['rating'].astype(float).values

    user_codes = pd.Categorical(train_df['user_id'], categories=train_users).codes.astype(np.int64)
    item_codes = pd.Categorical(train_df['item_id'], categories=all_items).codes.astype(np.int64)
    if (user_codes < 0).any() or (item_codes < 0).any():
        raise ValueError('ALS indexing failed: found unknown user_id/item_id codes during matrix build.')

    user_item = coo_matrix((vals, (user_codes, item_codes)), shape=(len(train_users), len(all_items))).tocsr()

    als = AlternatingLeastSquares(factors=50, iterations=15, regularization=0.01, random_state=RANDOM_STATE)
    als.fit(user_item)

    idx_to_item = {v: k for k, v in item_to_idx.items()}
    user_to_idx = {u: i for i, u in enumerate(train_users)}

    recs_als = {}
    for u in tqdm(users_test, desc='ALS Top-K'):
        uid = user_to_idx.get(u)
        if uid is None:
            recs_als[u] = all_items[:TOP_K]
            continue
        u_items = user_item[uid]
        rec_ids, _ = als.recommend(userid=0, user_items=u_items, N=TOP_K, filter_already_liked_items=True)
        recs_als[u] = [idx_to_item[i] for i in rec_ids if i in idx_to_item][:TOP_K]

    model_recs['ALS'] = recs_als
    model_rmse['ALS'] = np.nan
    del als, item_user, user_item
    gc.collect()
except Exception as e:
    print(f'ALS skipped due to error: {e}')
    model_recs['ALS'] = {u: all_items[:TOP_K] for u in users_test}
    model_rmse['ALS'] = np.nan

# ----- CBF: TF-IDF + Cosine -----
item_features_local = item_features[item_features['item_id'].isin(all_items)].drop_duplicates('item_id').reset_index(drop=True)
item_index = {it: i for i, it in enumerate(item_features_local['item_id'].tolist())}
sim = tfidf_item_similarity(item_features_local)
model_recs['TF-IDF'] = recommend_tfidf(train_df, users_test, all_items, item_index, sim, k=TOP_K)
model_rmse['TF-IDF'] = np.nan

del sim
gc.collect()

# ----- CBF: Logistic Regression (user-specific) -----
logreg_models = train_user_logistic_models(train_df, item_features_local)
model_recs['LogReg'] = recommend_user_logreg(logreg_models, users_test, user_seen, item_features_local, all_items, k=TOP_K)
model_rmse['LogReg'] = np.nan

del logreg_models
gc.collect()

# ----- CBF: Random Forest (metadata-driven) -----
rf, ue, ie = train_rf_regressor(train_df)
model_recs['RandomForest'] = recommend_rf(rf, ue, ie, users_test, user_seen, all_items, k=TOP_K)
model_rmse['RandomForest'] = np.nan

del rf, ue, ie
gc.collect()


## 7) Bias Metrics (Gini, ARP, coverage, Spearman, APP, long-tail, diversity, metadata entropy)

In [ ]:
rows = []
plot_rows = []

pct_map = train_popularity_percentile_map(train_df)

for model_name, recs in model_recs.items():
    rec_freq = recommendation_frequency(recs)
    all_freq = train_pop.copy().rename('train_pop').to_frame()
    all_freq['rec_freq'] = rec_freq
    all_freq['rec_freq'] = all_freq['rec_freq'].fillna(0.0)

    gini = gini_coefficient(all_freq['rec_freq'].values)
    arp = compute_arp(recs, pop_map)
    coverage = compute_catalog_coverage(recs, all_items)
    rho, _ = spearmanr(all_freq['train_pop'].values, all_freq['rec_freq'].values)
    app = avg_popularity_percentile(recs, pct_map)
    lt = long_tail_share(recs, pct_map, tail_cutoff=0.2)
    agg_div = aggregate_diversity_unique_items(recs)
    meta_h = metadata_token_entropy(recs, item_features_local)

    rows.append({
        'model': model_name,
        'gini': gini,
        'arp': arp,
        'catalog_coverage': coverage,
        'spearman_pop_vs_recfreq': rho,
        'avg_popularity_percentile': app,
        'long_tail_share_20pct': lt,
        'aggregate_diversity_unique_items': agg_div,
        'metadata_token_entropy_bits': meta_h,
        'rmse': model_rmse.get(model_name, np.nan),
    })

    tmp = all_freq.reset_index().rename(columns={'index': 'item_id'})
    tmp['model'] = model_name
    plot_rows.append(tmp)

metrics_df = pd.DataFrame(rows).sort_values('model')
metrics_df


## 8) Visualization: Popularity vs Recommendation Frequency (Log-Log)

In [ ]:
plot_df = pd.concat(plot_rows, ignore_index=True)
plot_df = plot_df[(plot_df['train_pop'] > 0) & (plot_df['rec_freq'] > 0)].copy()

fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharex=True, sharey=True)
axes = axes.flatten()
for ax, model_name in zip(axes, sorted(model_recs.keys())):
    d = plot_df[plot_df['model'] == model_name]
    ax.scatter(d['train_pop'], d['rec_freq'], alpha=0.35, s=12)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title(model_name)
    ax.set_xlabel('Global Popularity (train count)')
    ax.set_ylabel('Recommendation Frequency')

plt.tight_layout()
out_path = FIG_DIR / f'macro_bias_loglog_{DATASET_NAME}.png'
plt.savefig(out_path, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved figure: {out_path}')

# Extended metrics (bar panels)
ext_cols = [
    'avg_popularity_percentile',
    'long_tail_share_20pct',
    'aggregate_diversity_unique_items',
    'metadata_token_entropy_bits',
]
fig2, axes2 = plt.subplots(2, 2, figsize=(12, 10))
axes2 = axes2.flatten()
for ax, col in zip(axes2, ext_cols):
    dfp = metrics_df.sort_values('model')
    sns.barplot(data=dfp, x='model', y=col, ax=ax)
    ax.set_title(col.replace('_', ' '))
    ax.tick_params(axis='x', rotation=25)
plt.suptitle(f'Extended macro-bias metrics ({DATASET_NAME})', y=1.02)
plt.tight_layout()
out2 = FIG_DIR / f'macro_bias_extended_metrics_{DATASET_NAME}.png'
plt.savefig(out2, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved figure: {out2}')

# ECDF: log10(train popularity) for catalog vs recommended slots per model
def ecdf_vals(x):
    x = np.sort(np.asarray(x, dtype=float))
    y = np.arange(1, len(x) + 1) / max(1, len(x))
    return x, y

fig3, ax3 = plt.subplots(figsize=(10, 6))
train_vals = np.log10(train_pop.values.astype(float) + 1.0)
x0, y0 = ecdf_vals(train_vals)
ax3.plot(x0, y0, label='Train catalog (items)', color='black', linewidth=2.2, linestyle='--')
colors = sns.color_palette('husl', n_colors=max(6, len(model_recs)))
for i, (name, recs) in enumerate(sorted(model_recs.items())):
    vals = [np.log10(float(train_pop.get(it, 0.0)) + 1.0) for items in recs.values() for it in items]
    if not vals:
        continue
    x, y = ecdf_vals(np.array(vals))
    ax3.plot(x, y, label=name, color=colors[i % len(colors)], linewidth=1.6)
ax3.set_xlabel('log10(train interactions + 1)')
ax3.set_ylabel('ECDF')
ax3.set_title(f'Popularity distribution: train vs recommendations ({DATASET_NAME})')
ax3.legend(loc='lower right', fontsize=8)
ax3.grid(True, alpha=0.3)
plt.tight_layout()
out3 = FIG_DIR / f'macro_bias_popularity_cdf_{DATASET_NAME}.png'
plt.savefig(out3, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved figure: {out3}')


## 9) Save Results

In [ ]:
out_metrics = RES_DIR / f'macro_bias_metrics_{DATASET_NAME}.csv'
metrics_df.to_csv(out_metrics, index=False)
print(f'Saved metrics: {out_metrics}')

# Optional snapshot of top-10 for first 10 users per model
sample = []
for m, recs in model_recs.items():
    for u in list(recs.keys())[:10]:
        sample.append({'model': m, 'user_id': u, 'top10': recs[u]})

pd.DataFrame(sample).to_json(RES_DIR / f'macro_bias_top10_sample_{DATASET_NAME}.json', orient='records', indent=2)
print('Saved top-10 sample json.')
